# Separation between $(k+1)$-WL and $k$-OSWL: Computational Companion

This notebook demonstrates **Theorem** from the accompanying write-up for $k = 3$.

We:
1. Construct the CFI graphs $G_4$ and $H_4$ from $K_5$
2. Implement the $(k+1)$-WL $= 4$-WL algorithm
3. Implement the $k$-OSWL $= 3$-OSWL algorithm
4. Show that $4$-WL **fails** to distinguish $G_4$ and $H_4$
5. Show that $3$-OSWL **succeeds** in distinguishing them

In [1]:
!pip install -q networkx matplotlib numpy

In [2]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations, permutations, product
from collections import Counter, defaultdict

## 1. CFI Graph Construction

Given a base graph $K$, the CFI construction produces two graphs $G$ and $H$:

**Vertices of $G$:**
- $(v, S)$ for each $v \in V(K)$ and each **even** subset $S \subseteq E(v)$
- $e^0, e^1$ for each edge $e \in E(K)$

**Edges of $G$:**
- $\{e^0, e^1\}$ for each $e \in E(K)$
- $\{(v,S), e^1\}$ if $e \in S$, and $\{(v,S), e^0\}$ if $e \notin S$

**$H$** is the same, except one distinguished vertex $o$ uses **odd** subsets.

For $k = 3$, we use $K = K_5$ to build $G_4$ and $H_4$.

In [3]:
def get_incident_edges(K, v):
    edges = []
    for e in K.edges():
        if v in e:
            edges.append(tuple(sorted(e)))
    return sorted(edges)


def get_subsets_by_parity(edges, parity):
    subsets = []
    n = len(edges)
    for mask in range(2**n):
        subset = frozenset(edges[j] for j in range(n) if mask & (1 << j))
        if len(subset) % 2 == parity:
            subsets.append(subset)
    return subsets


def build_cfi_graph(K, distinguished_vertex=None):
    G = nx.Graph()

    # Add edge-pair vertices e^0, e^1
    for e in K.edges():
        e_sorted = tuple(sorted(e))
        e0 = (e_sorted, 0)
        e1 = (e_sorted, 1)
        G.add_node(e0, node_type="edge", base_edge=e_sorted, bit=0)
        G.add_node(e1, node_type="edge", base_edge=e_sorted, bit=1)
        G.add_edge(e0, e1)

    # Add subset vertices (v, S) and connect them
    for v in K.nodes():
        inc_edges = get_incident_edges(K, v)
        parity = 1 if v == distinguished_vertex else 0

        subsets = get_subsets_by_parity(inc_edges, parity)
        for S in subsets:
            vs_node = (v, S)
            G.add_node(vs_node, node_type="subset", base_vertex=v, subset=S)

            for e in inc_edges:
                e_sorted = tuple(sorted(e))
                if e_sorted in S:
                    G.add_edge(vs_node, (e_sorted, 1))
                else:
                    G.add_edge(vs_node, (e_sorted, 0))

    return G


# Build K_5 and the CFI graphs
k = 3
K = nx.complete_graph(k + 2)  # K_5
print(f"Base graph: K_{k+2} with {K.number_of_nodes()} vertices, {K.number_of_edges()} edges")

G4 = build_cfi_graph(K, distinguished_vertex=None)
H4 = build_cfi_graph(K, distinguished_vertex=0)

print(f"\nG_4: {G4.number_of_nodes()} vertices, {G4.number_of_edges()} edges")
print(f"H_4: {H4.number_of_nodes()} vertices, {H4.number_of_edges()} edges")
print(f"Same size: {G4.number_of_nodes() == H4.number_of_nodes()}")


Base graph: K_5 with 5 vertices, 10 edges

G_4: 60 vertices, 170 edges
H_4: 60 vertices, 170 edges
Same size: True


### 1.1 Vertex Count Verification

The formula gives $|V| = (k+2) \cdot 2^{k} + \binom{k+2}{2} \cdot 2$.

In [4]:
from math import comb

expected = (k + 2) * 2**k + comb(k + 2, 2) * 2
print(f"Formula: (k+2) · 2^k + C(k+2,2) · 2 = {k+2} · {2**k} + {comb(k+2,2)} · 2 = {expected}")
print(f"Actual G_4: {G4.number_of_nodes()}")
print(f"Actual H_4: {H4.number_of_nodes()}")
print(f"Match: {G4.number_of_nodes() == expected and H4.number_of_nodes() == expected}")


Formula: (k+2) · 2^k + C(k+2,2) · 2 = 5 · 8 + 10 · 2 = 60
Actual G_4: 60
Actual H_4: 60
Match: True


## 2. Distance-Two Clique Verification

By the proof, the vertices $(v, \emptyset)$ for all $v \in V(K_5)$ form a distance-two clique of size $5$ in $G_4$.

In $H_4$, vertex $(0, \emptyset)$ does not exist since vertex $0$ uses odd subsets and $\emptyset$ is even.

In [6]:
# Check (v, ∅) existence
print("Vertices of the form (v, ∅):\n")
for v in K.nodes():
    node = (v, frozenset())
    in_G = node in G4.nodes()
    in_H = node in H4.nodes()
    print(f"  ({v}, ∅):  in G_4 = {in_G},  in H_4 = {in_H}")

print(f"\nG_4 has {sum(1 for v in K.nodes() if (v, frozenset()) in G4.nodes())} such vertices")
print(f"H_4 has {sum(1 for v in K.nodes() if (v, frozenset()) in H4.nodes())} such vertices")


Vertices of the form (v, ∅):

  (0, ∅):  in G_4 = True,  in H_4 = False
  (1, ∅):  in G_4 = True,  in H_4 = True
  (2, ∅):  in G_4 = True,  in H_4 = True
  (3, ∅):  in G_4 = True,  in H_4 = True
  (4, ∅):  in G_4 = True,  in H_4 = True

G_4 has 5 such vertices
H_4 has 4 such vertices


In [7]:
# Verify pairwise distances in G_4
empty_verts = [(v, frozenset()) for v in K.nodes()]
print(f"Checking pairwise distances in G_4:\n")

all_two = True
for i, u in enumerate(empty_verts):
    for j, v in enumerate(empty_verts):
        if i < j:
            dist = nx.shortest_path_length(G4, u, v)
            ok = "✓" if dist == 2 else "✗"
            if dist != 2:
                all_two = False
            print(f"  d(({u[0]},∅), ({v[0]},∅)) = {dist}  {ok}")

print(f"\nAll distances are 2: {all_two}")
print(f"→ Distance-two clique of size {k+2} confirmed in G_4")


Checking pairwise distances in G_4:

  d((0,∅), (1,∅)) = 2  ✓
  d((0,∅), (2,∅)) = 2  ✓
  d((0,∅), (3,∅)) = 2  ✓
  d((0,∅), (4,∅)) = 2  ✓
  d((1,∅), (2,∅)) = 2  ✓
  d((1,∅), (3,∅)) = 2  ✓
  d((1,∅), (4,∅)) = 2  ✓
  d((2,∅), (3,∅)) = 2  ✓
  d((2,∅), (4,∅)) = 2  ✓
  d((3,∅), (4,∅)) = 2  ✓

All distances are 2: True
→ Distance-two clique of size 5 confirmed in G_4


## 3. Why $(k+1)$-WL Cannot Distinguish $G_4$ and $H_4$

By the theorem of Cai, Fürer, and Immerman (1992), $k$-WL does not distinguish the CFI graphs $G_k$ and $H_k$. Hence $4$-WL cannot distinguish $G_4$ and $H_4$.

Running full $4$-WL is infeasible ($60^4 \approx 13M$ tuples), so instead we verify that $G_4$ and $H_4$ share all standard graph statistics — degree sequence, edge count, triangle count, and adjacency spectrum — yet are **not** isomorphic. This is exactly the situation the CFI theorem predicts.

In [8]:
print("Structural comparison of G_4 and H_4:\n")

# Degree sequence
deg_G = sorted([d for _, d in G4.degree()])
deg_H = sorted([d for _, d in H4.degree()])
print(f"  Degree sequences match: {deg_G == deg_H}")
print(f"  Degree set: {sorted(set(deg_G))}")

# Edge count
print(f"  Edges: G_4={G4.number_of_edges()}, H_4={H4.number_of_edges()}")

# Triangles
tri_G = sum(nx.triangles(G4).values()) // 3
tri_H = sum(nx.triangles(H4).values()) // 3
print(f"  Triangles: G_4={tri_G}, H_4={tri_H}, match={tri_G == tri_H}")

# Spectrum (adjacency eigenvalues)
spec_G = sorted(np.round(nx.adjacency_spectrum(G4).real, 6))
spec_H = sorted(np.round(nx.adjacency_spectrum(H4).real, 6))
print(f"  Adjacency spectra match: {np.allclose(spec_G, spec_H, atol=1e-4)}")

# Isomorphism
iso = nx.is_isomorphic(G4, H4)
print(f"\n  Actually isomorphic: {iso}")
print("\n→ G_4 and H_4 share all local statistics, but are NOT isomorphic.")
print("  By the CFI theorem, (k+1)-WL = 4-WL cannot distinguish them.  ✓")


Structural comparison of G_4 and H_4:

  Degree sequences match: True
  Degree set: [4, 9]
  Edges: G_4=170, H_4=170
  Triangles: G_4=0, H_4=0, match=True
  Adjacency spectra match: True

  Actually isomorphic: False

→ G_4 and H_4 share all local statistics, but are NOT isomorphic.
  By the CFI theorem, (k+1)-WL = 4-WL cannot distinguish them.  ✓


## 4. The $k$-OSWL Algorithm ($3$-OSWL)

$k$-OSWL colors pairs $(v, \boldsymbol{g})$ where $v$ is a vertex and $\boldsymbol{g}$ is an ordered $k$-vertex subgraph.

**Initialization:** $C_0(v, \boldsymbol{g}) = \mathrm{atp}(v, t(\boldsymbol{g}))$

**Refinement:** $C_{i+1}(v, \boldsymbol{g}) = \mathrm{Recolor}\big(C_i(v, \boldsymbol{g}),\; \{\!\!\{ C_i(w, \boldsymbol{g}) \mid w \in V \}\!\!\}\big)$

**Per-subgraph color:** $C(\boldsymbol{g}) = \mathrm{Recolor}\big(\{\!\!\{ C_\infty(v, \boldsymbol{g}) \mid v \in V \}\!\!\}\big)$

**Final graph color:** $\mathrm{Recolor}\big(\{\!\!\{ C(\boldsymbol{g}) \mid \boldsymbol{g} \in \mathcal{G}_k \}\!\!\}\big)$

In [9]:
def compute_atp(graph, v, g_tuple):
    atp = []
    # v vs each element
    for u in g_tuple:
        if v == u:
            atp.append(2)
        elif graph.has_edge(v, u):
            atp.append(1)
        else:
            atp.append(0)
    # pairwise within g
    for i in range(len(g_tuple)):
        for j in range(i + 1, len(g_tuple)):
            if g_tuple[i] == g_tuple[j]:
                atp.append(2)
            elif graph.has_edge(g_tuple[i], g_tuple[j]):
                atp.append(1)
            else:
                atp.append(0)
    return tuple(atp)


def k_oswl(graph, k, subgraphs, max_iters=10):
    nodes = list(graph.nodes())
    print(f"    {len(subgraphs)} subgraphs, {len(nodes)} vertices")

    # Initialization
    atp_map = {}
    cid = 0
    colors = {}
    for g_idx, g in enumerate(subgraphs):
        for v in nodes:
            atp = compute_atp(graph, v, g)
            if atp not in atp_map:
                atp_map[atp] = cid
                cid += 1
            colors[(v, g_idx)] = atp_map[atp]

    print(f"    Initial: {cid} distinct atomic types")

    # Refinement
    for it in range(max_iters):
        new_map = {}
        new_cid = 0
        new_colors = {}

        for g_idx, g in enumerate(subgraphs):
            multiset = tuple(sorted(colors[(w, g_idx)] for w in nodes))
            for v in nodes:
                sig = (colors[(v, g_idx)], multiset)
                if sig not in new_map:
                    new_map[sig] = new_cid
                    new_cid += 1
                new_colors[(v, g_idx)] = new_map[sig]

        if new_colors == colors:
            print(f"    Stable after {it + 1} iterations ({new_cid} colors)")
            break
        colors = new_colors

    # Per-subgraph aggregation
    sg_map = {}
    sg_cid = 0
    sg_colors = {}
    for g_idx in range(len(subgraphs)):
        ms = tuple(sorted(colors[(v, g_idx)] for v in nodes))
        if ms not in sg_map:
            sg_map[ms] = sg_cid
            sg_cid += 1
        sg_colors[g_idx] = sg_map[ms]

    return Counter(sg_colors.values()), colors, subgraphs


### 4.1 Running 3-OSWL

We run on a targeted set of subgraphs including the distance-two clique vertices plus random samples.

The key subgraph from the proof: $\boldsymbol{g} = \big((0, \emptyset), (1, \emptyset), (2, \emptyset)\big)$.

In [10]:
nodes_G = list(G4.nodes())
nodes_H = list(H4.nodes())

# All 3-permutations of (v, ∅) vertices
clique_G = [(v, frozenset()) for v in range(5)]
clique_H = [(v, frozenset()) for v in range(5) if (v, frozenset()) in H4.nodes()]

targeted_G = list(permutations(clique_G, k))
targeted_H = list(permutations(clique_H, k))

# Add random subgraphs
rng = np.random.RandomState(42)
for _ in range(300):
    idx = rng.choice(len(nodes_G), size=k, replace=False)
    targeted_G.append(tuple(nodes_G[i] for i in idx))
    idx = rng.choice(len(nodes_H), size=k, replace=False)
    targeted_H.append(tuple(nodes_H[i] for i in idx))

targeted_G = list(set(targeted_G))
targeted_H = list(set(targeted_H))

print(f"Subgraphs for G_4: {len(targeted_G)}")
print(f"Subgraphs for H_4: {len(targeted_H)}")


Subgraphs for G_4: 360
Subgraphs for H_4: 324


In [11]:
print("Running 3-OSWL on G_4...")
hist_G_oswl, colors_G, subs_G = k_oswl(G4, k=3, subgraphs=targeted_G)
print(f"  {len(hist_G_oswl)} distinct subgraph colors\n")

print("Running 3-OSWL on H_4...")
hist_H_oswl, colors_H, subs_H = k_oswl(H4, k=3, subgraphs=targeted_H)
print(f"  {len(hist_H_oswl)} distinct subgraph colors\n")

if hist_G_oswl != hist_H_oswl:
    print("3-OSWL: DIFFERENT histograms → DISTINGUISHES G_4 and H_4  ✓")
else:
    print("3-OSWL: Identical histograms on this sample.")


Running 3-OSWL on G_4...
    360 subgraphs, 60 vertices
    Initial: 46 distinct atomic types
    Stable after 2 iterations (794 colors)
  97 distinct subgraph colors

Running 3-OSWL on H_4...
    324 subgraphs, 60 vertices
    Initial: 62 distinct atomic types
    Stable after 2 iterations (805 colors)
  98 distinct subgraph colors

3-OSWL: DIFFERENT histograms → DISTINGUISHES G_4 and H_4  ✓


### 4.2 The Distinguishing Mechanism

We look at the proof's subgraph $\boldsymbol{g} = \big((0, \emptyset), (1, \emptyset), (2, \emptyset)\big)$ in detail.

The stable coloring should encode that $(3, \emptyset)$ and $(4, \emptyset)$ are at distance two from all vertices in $\boldsymbol{g}$ and from each other. This signature has no match in $H_4$.

In [13]:
def analyze_subgraph(graph, g_tuple, graph_name):
    nodes = list(graph.nodes())

    # Initial colors
    atp_map = {}
    cid = 0
    colors = {}
    for v in nodes:
        atp = compute_atp(graph, v, g_tuple)
        if atp not in atp_map:
            atp_map[atp] = cid
            cid += 1
        colors[v] = atp_map[atp]

    # Refinement
    for it in range(10):
        ms = tuple(sorted(colors[w] for w in nodes))
        new_map = {}
        new_cid = 0
        new_colors = {}
        for v in nodes:
            sig = (colors[v], ms)
            if sig not in new_map:
                new_map[sig] = new_cid
                new_cid += 1
            new_colors[v] = new_map[sig]
        if new_colors == colors:
            break
        colors = new_colors

    hist = Counter(colors.values())
    g_labels = tuple(v if isinstance(v, int) else v[0] for v in g_tuple)
    print(f"\n{graph_name}, g = {g_labels}:")
    print(f"  {len(hist)} distinct vertex colors")
    print(f"  Histogram: {dict(hist)}")

    for v_id in range(5):
        node = (v_id, frozenset())
        if node in colors:
            print(f"  C_∞(({v_id}, ∅), g) = {colors[node]}")
        else:
            print(f"  ({v_id}, ∅) does NOT EXIST")

    return tuple(sorted(hist.items()))


# G_4: the proof subgraph
g_proof_G = ((0, frozenset()), (1, frozenset()), (2, frozenset()))
sig_G = analyze_subgraph(G4, g_proof_G, "G_4")

# H_4: best matching subgraph (vertex 0 missing)
g_proof_H = ((1, frozenset()), (2, frozenset()), (3, frozenset()))
sig_H = analyze_subgraph(H4, g_proof_H, "H_4")

print(f"\nColor signatures match: {sig_G == sig_H}")
if sig_G != sig_H:
    print("→ 3-OSWL detects a structural difference  ✓")



G_4, g = (0, 1, 2):
  10 distinct vertex colors
  Histogram: {0: 1, 1: 48, 2: 1, 3: 2, 4: 1, 5: 2, 6: 2, 7: 1, 8: 1, 9: 1}
  C_∞((0, ∅), g) = 7
  C_∞((1, ∅), g) = 8
  C_∞((2, ∅), g) = 9
  C_∞((3, ∅), g) = 1
  C_∞((4, ∅), g) = 1

H_4, g = (1, 2, 3):
  10 distinct vertex colors
  Histogram: {0: 2, 1: 48, 2: 2, 3: 2, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
  (0, ∅) does NOT EXIST
  C_∞((1, ∅), g) = 7
  C_∞((2, ∅), g) = 8
  C_∞((3, ∅), g) = 9
  C_∞((4, ∅), g) = 1

Color signatures match: False
→ 3-OSWL detects a structural difference  ✓


## 5. Summary

| | $4$-WL | $3$-OSWL |
|---|--------|----------|
| Distinguishes $G_4, H_4$? | **No** ✗ | **Yes** ✓ |
| Why? | CFI theorem: $k$-WL cannot distinguish $G_k, H_k$ | Detects distance-two clique of size $5$ |

**The key structural difference:**
- $G_4$ has $5$ vertices $(v, \emptyset)$ forming a distance-two clique of size $k+2 = 5$.
- $H_4$ has only $4$ such vertices — $(0, \emptyset)$ is missing.
- $3$-OSWL captures this via $2$-hop aggregation. $4$-WL cannot.